<a href="https://colab.research.google.com/github/KasiR07/Calories-Burnt-Prediction/blob/main/Calories_burnt_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Calories Burnt Prediction

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn import metrics
from sklearn.svm import SVC
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor

import warnings
warnings.filterwarnings('ignore')

## Load Data

Merge Datasets

In [ ]:
df1 = pd.read_csv('/content/sample_data/exercise.csv')
df2 = pd.read_csv('/content/sample_data/calories.csv')


In [ ]:
print("DF1 Columns:", df1.columns.tolist())
print("DF2 Columns:", df2.columns.tolist())

In [ ]:
df = pd.concat([df1, df2], axis=1)

Data Preprocessing

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.size

In [ ]:
df.info()

In [ ]:
df.describe()

###  EDA

In [ ]:
sb.scatterplot(x='Height', y='Weight', data=df)
plt.show()


In [ ]:
features = ['Age', 'Height', 'Weight', 'Duration']

plt.subplots(figsize=(15, 10))
for i, col in enumerate(features):
    plt.subplot(2, 2, i + 1)
    x = df.sample(1000)
    sb.scatterplot(x=col, y='Calories', data=x)
plt.tight_layout()
plt.show()


As expected higher is the duration of the workout higher will be the calories burnt. But except for that, we cannot observe any such relation between calories burnt and height or weight features.

In [ ]:
features = df.select_dtypes(include='float').columns

plt.subplots(figsize=(15, 10))
for i, col in enumerate(features):
    plt.subplot(2, 3, i + 1)
    sb.distplot(df[col])
plt.tight_layout()
plt.show()

The distribution of the continuous features follows close to normal distribution except for some features like Body_Temp and Calories.

In [ ]:
df.replace({'male': 0, 'female': 1},
           inplace=True)
df.head()

In [ ]:
plt.figure(figsize=(8, 8))
sb.heatmap(df.corr() > 0.9,
           annot=True,
           cbar=False)
plt.show()

There is a serious problem of data leakage as there is a feature that is highly correlated with the target column which is calories.

In [ ]:
to_remove = ['Weight', 'Duration']
df.drop(to_remove, axis=1, inplace=True)

## Model Training

In [ ]:
features = df.drop(['User_ID', 'Calories'], axis=1)
target = df['Calories'].values

X_train, X_val,\
    Y_train, Y_val = train_test_split(features, target,
                                      test_size=0.1,
                                      random_state=22)
X_train.shape, X_val.shape

In [ ]:
# Normalizing the features for stable and fast training.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

In [ ]:
from sklearn.metrics import mean_absolute_error as mae
models = [LinearRegression(), XGBRegressor(),
          Lasso(), RandomForestRegressor(), Ridge()]

for i in range(5):
    models[i].fit(X_train, Y_train)

    print(f'{models[i]} : ')

    train_preds = models[i].predict(X_train)
    print('Training Error : ', mae(Y_train, train_preds))

    val_preds = models[i].predict(X_val)
    print('Validation Error : ', mae(Y_val, val_preds))
    print()